In [ ]:
# Complete PyTorch pipeline (optimized for limited GPU memory and better output)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm
import re

# Dataset class with filename-matching
class Img2ImgDataset(Dataset):
    def __init__(self, input_path, target_path):
        input_files = sorted(glob(os.path.join(input_path, '*.tif')))
        target_files = sorted(glob(os.path.join(target_path, '*.tif')))

        # Create dicts based on numeric ID
        input_dict = {re.findall(r'(\d+)', os.path.basename(f))[0]: f for f in input_files}
        target_dict = {re.findall(r'(\d+)', os.path.basename(f))[0]: f for f in target_files}

        common_keys = sorted(set(input_dict.keys()) & set(target_dict.keys()))
        if not common_keys:
            raise ValueError("No matching image pairs found based on numeric filename IDs.")

        self.input_files = [input_dict[k] for k in common_keys]
        self.target_files = [target_dict[k] for k in common_keys]

        self.transform = transforms.Compose([
            transforms.CenterCrop(256),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        input_img = Image.open(self.input_files[idx])
        target_img = Image.open(self.target_files[idx])
        return self.transform(input_img), self.transform(target_img)

# U-Net model
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        for f in features:
            self.downs.append(self._block(in_channels, f))
            in_channels = f

        self.bottleneck = self._block(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.ups.append(self._block(f * 2, f))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(len(self.ups)):
            x = self.upconvs[i](x)
            if x.shape != skips[i].shape:
                x = F.interpolate(x, size=skips[i].shape[2:])
            x = torch.cat((skips[i], x), dim=1)
            x = self.ups[i](x)
        return self.final_conv(x)

# Training
def train_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += loss_fn(pred, y).item() * x.size(0)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader.dataset):.4f} | Val Loss: {val_loss/len(val_loader.dataset):.4f}", flush=True)

# Visualization
def visualize_predictions(model, dataset, device, n=3):
    model.eval()
    model.to(device)
    with torch.no_grad():
        for i in range(min(n, len(dataset))):
            x, y = dataset[i]
            x = x.unsqueeze(0).to(device)
            pred = model(x).squeeze().cpu().numpy()

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))
            axs[0].imshow(x.squeeze().cpu().numpy(), cmap='gray')
            axs[0].set_title("Input")
            axs[1].imshow(y.squeeze().numpy(), cmap='gray')
            axs[1].set_title("Ground Truth")
            axs[2].imshow(pred, cmap='gray')
            axs[2].set_title("Prediction")
            for ax in axs: ax.axis("off")
            plt.show()

# Main (no download step)
if __name__ == "__main__":
    base_path = "/home/hpc/tovl/tovl103v/anmol_bai/04_image_to_image/dataset/"
    input_path = os.path.join(base_path, "phase_contrast")
    target_path = os.path.join(base_path, "projected_thickness")

    print("Input folder exists:", os.path.exists(input_path))
    print("Target folder exists:", os.path.exists(target_path))
    print("Sample input files:", glob(os.path.join(input_path, '*.tif'))[:3])
    print("Sample target files:", glob(os.path.join(target_path, '*.tif'))[:3])

    dataset = Img2ImgDataset(input_path, target_path)

    if len(dataset) == 0:
        raise ValueError("Dataset appears to be empty. Please check the image folder paths.")

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    if train_size == 0 or val_size == 0:
        raise ValueError("Train or validation split resulted in zero samples. Add more images.")

    train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet()

    print(f"Using device: {device} | Dataset size: {len(dataset)}")
    train_model(model, train_loader, val_loader, device)
    visualize_predictions(model, val_ds, device)

In [ ]:
torch.save(model.state_dict(), "unet_trained_weights.pth")


In [ ]:
import random

def visualize_random_predictions(model, dataset, device, n=20):
    model.eval()
    model.to(device)
    indices = random.sample(range(len(dataset)), n)

    with torch.no_grad():
        for idx in indices:
            x, y = dataset[idx]
            x_tensor = x.unsqueeze(0).to(device)
            pred = model(x_tensor).squeeze().cpu().numpy()

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))
            axs[0].imshow(x.squeeze().cpu().numpy(), cmap='gray')
            axs[0].set_title("Input")
            axs[1].imshow(y.squeeze().numpy(), cmap='gray')
            axs[1].set_title("Ground Truth")
            axs[2].imshow(pred, cmap='gray')
            axs[2].set_title("Prediction")
            for ax in axs: ax.axis("off")
            plt.show()

# 🔁 Load model + weights and visualize
model = UNet()
model.load_state_dict(torch.load("unet_trained_weights.pth", map_location=device))
model.to(device)

visualize_random_predictions(model, dataset, device, n=10)


In [ ]:
import torch
from torch.utils.data import DataLoader

# Step 1: Reuse your model class definition
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        for f in features:
            self.downs.append(self._block(in_channels, f))
            in_channels = f

        self.bottleneck = self._block(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.ups.append(self._block(f * 2, f))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(len(self.ups)):
            x = self.upconvs[i](x)
            if x.shape != skips[i].shape:
                x = F.interpolate(x, size=skips[i].shape[2:])
            x = torch.cat((skips[i], x), dim=1)
            x = self.ups[i](x)
        return self.final_conv(x)

# Step 2: Reload dataset and dataloaders
# (Assume dataset, train_loader, val_loader already exist, otherwise re-define them)

# Step 3: Reload model with saved weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet()
model.load_state_dict(torch.load("unet_trained_weights.pth", map_location=device))
model.to(device)

# Step 4: Define training function again
def train_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += loss_fn(pred, y).item() * x.size(0)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader.dataset):.4f} | Val Loss: {val_loss/len(val_loader.dataset):.4f}", flush=True)

    # Save updated weights
    torch.save(model.state_dict(), "unet_trained_weights.pth")
    print("✅ Updated weights saved to unet_trained_weights.pth")

# Step 5: Continue training
train_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4)
